# Late Transaction Revenue Correction

## Phase 1 - Data Engineering

This project solves delayed transaction reporting problems using Databricks,
PySpark, Delta Lake, and Medallion Architecture.

The pipeline consists of:

1. Bronze Layer — Raw transaction ingestion using Auto Loader
2. Silver Layer — Data cleaning, validation, and deduplication
3. Gold Layer — Daily revenue reporting
4. Late Transaction Detection
5. Affected Historical Date Identification
6. Selective Revenue Recalculation
7. Delta Lake MERGE for historical correction
8. Data Quality Validation
9. Watermark-based incremental processing

The final corrected revenue data will later be exposed through a FastAPI
backend and an HTML, CSS, and JavaScript dashboard.

## Step 1 — Project Configuration

In this section, we define the locations required for the pipeline.

The source folder contains incoming transaction files.

The Bronze, Silver, and Gold folders will contain the corresponding
Delta Lake data.

Checkpoint locations are used by Structured Streaming and Auto Loader
to keep track of processed files.

In [0]:
# PROJECT CONFIGURATION

# Base location of our Unity Catalog Volume
BASE_PATH = "/Volumes/workspace/default/late_transaction"

# Folder containing incoming raw CSV files
SOURCE_PATH = f"{BASE_PATH}/source"

# Bronze layer location
BRONZE_PATH = f"{BASE_PATH}/bronze"

# Silver layer location
SILVER_PATH = f"{BASE_PATH}/silver"

# Gold layer location
GOLD_PATH = f"{BASE_PATH}/gold"

# Checkpoint location for Bronze Auto Loader
BRONZE_CHECKPOINT = f"{BASE_PATH}/checkpoints/bronze"

# Print the paths so we can verify our configuration
print("Base Path:", BASE_PATH)
print("Source Path:", SOURCE_PATH)
print("Bronze Path:", BRONZE_PATH)
print("Silver Path:", SILVER_PATH)
print("Gold Path:", GOLD_PATH)
print("Bronze Checkpoint:", BRONZE_CHECKPOINT)

Base Path: /Volumes/workspace/default/late_transaction
Source Path: /Volumes/workspace/default/late_transaction/source
Bronze Path: /Volumes/workspace/default/late_transaction/bronze
Silver Path: /Volumes/workspace/default/late_transaction/silver
Gold Path: /Volumes/workspace/default/late_transaction/gold
Bronze Checkpoint: /Volumes/workspace/default/late_transaction/checkpoints/bronze


# Step 2 — Prepare Source Data

The source layer contains the incoming raw transaction CSV files.

In this step, we create the source directory and verify that the input transaction file is available before starting the Bronze ingestion process.

The source data will be processed by Auto Loader in the Bronze layer.

In [0]:
# STEP 2 — PREPARE SOURCE DATA

# Create the source directory if it does not already exist.
dbutils.fs.mkdirs(SOURCE_PATH)

# Display the files currently available in the source directory.
print("Source directory:", SOURCE_PATH)
display(dbutils.fs.ls(SOURCE_PATH))

Source directory: /Volumes/workspace/default/late_transaction/source


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/late_transaction/source/sales_2000_rows.csv,sales_2000_rows.csv,70585,1786356019000


## Step 2A - Bronze Layer Ingestion

In this step, incoming CSV transaction files are ingested into the Bronze layer using Databricks Auto Loader.

The raw data is stored in Delta Lake format without applying business transformations.

Auto Loader uses a checkpoint and schema location to track processed files and maintain reliable incremental ingestion.

In [0]:
# RESET BRONZE INGESTION STATE

dbutils.fs.rm(BRONZE_PATH, True)
dbutils.fs.rm(BRONZE_CHECKPOINT, True)
dbutils.fs.rm(f"{BASE_PATH}/checkpoints/bronze_schema", True)

print("Bronze target and checkpoint state cleared.")

Bronze target and checkpoint state cleared.


In [0]:
# BRONZE LAYER INGESTION

# Create required directories
dbutils.fs.mkdirs(BRONZE_PATH)
dbutils.fs.mkdirs(BRONZE_CHECKPOINT)
dbutils.fs.mkdirs(f"{BASE_PATH}/checkpoints/bronze_schema")

# Read incoming CSV files using Auto Loader
bronze_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"{BASE_PATH}/checkpoints/bronze_schema")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(SOURCE_PATH)
)

# Write raw data to Bronze as Delta
bronze_query = (
    bronze_stream.writeStream
        .format("delta")
        .option("checkpointLocation", BRONZE_CHECKPOINT)
        .option("path", BRONZE_PATH)
        .trigger(availableNow=True)
        .start()
)

bronze_query.awaitTermination()

print("Bronze ingestion completed successfully.")

Bronze ingestion completed successfully.


## Step 3 — Validate Bronze Layer

In this step, we validate the Bronze Delta table created from the incoming transaction files.

The validation checks:
- Whether the Bronze Delta path exists
- Total number of ingested records
- Bronze table schema
- Sample transaction records

This confirms that the raw transaction data has been successfully stored in the Bronze layer.

In [0]:
# STEP 3 — VERIFY BRONZE DELTA OUTPUT

print("Bronze path contents:")
display(dbutils.fs.ls(BRONZE_PATH))

print("Delta log contents:")
display(dbutils.fs.ls(f"{BRONZE_PATH}/_delta_log"))

Bronze path contents:


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/late_transaction/bronze/_delta_log/,_delta_log/,0,1786364487184
dbfs:/Volumes/workspace/default/late_transaction/bronze/part-00000-baacd23f-67fe-4d7c-8bf9-cd63baa90677.c000.snappy.parquet,part-00000-baacd23f-67fe-4d7c-8bf9-cd63baa90677.c000.snappy.parquet,28088,1786364333000


Delta log contents:


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/late_transaction/bronze/_delta_log/00000000000000000000.crc,00000000000000000000.crc,2379,1786364329000
dbfs:/Volumes/workspace/default/late_transaction/bronze/_delta_log/00000000000000000000.json,00000000000000000000.json,1483,1786364329000
dbfs:/Volumes/workspace/default/late_transaction/bronze/_delta_log/00000000000000000001.crc,00000000000000000001.crc,3217,1786364334000
dbfs:/Volumes/workspace/default/late_transaction/bronze/_delta_log/00000000000000000001.json,00000000000000000001.json,1598,1786364334000
dbfs:/Volumes/workspace/default/late_transaction/bronze/_delta_log/_staged_commits/,_staged_commits/,0,1786364487960


# STEP 4 — SILVER LAYER

The Silver layer cleans and transforms the raw Bronze transaction data.

The following operations are performed:

- Clean string values
- Convert transaction dates into proper date types
- Convert transaction amounts into numeric values
- Remove invalid records
- Remove duplicate transactions
- Identify late-arriving transactions
- Store the cleaned data as a Delta table

In [0]:
# STEP 4 — CREATE SILVER LAYER

from pyspark.sql.functions import col, trim, to_date, row_number, datediff, when
from pyspark.sql.window import Window

# Create Silver path
SILVER_PATH = f"{BASE_PATH}/silver"

dbutils.fs.mkdirs(SILVER_PATH)

# Read Bronze Delta table
bronze_df = (
    spark.read
    .format("delta")
    .load(BRONZE_PATH)
)

print("Bronze data loaded successfully.")

# Clean and transform data
silver_df = (
    bronze_df
    .withColumn("txn_id", trim(col("txn_id")))
    .withColumn("user_id", trim(col("user_id")))
    .withColumn("txn_date", to_date(trim(col("txn_date"))))
    .withColumn("amount", col("amount").cast("double"))
    .withColumn("ingestion_date", to_date(trim(col("ingestion_date"))))
)

# Remove invalid records
silver_df = silver_df.filter(
    col("txn_id").isNotNull() &
    col("user_id").isNotNull() &
    col("txn_date").isNotNull() &
    col("amount").isNotNull() &
    (col("amount") >= 0) &
    col("ingestion_date").isNotNull()
)

# Remove duplicate transactions
window_spec = Window.partitionBy("txn_id").orderBy(
    col("ingestion_date").desc()
)

silver_df = (
    silver_df
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

# Identify late-arriving transactions
silver_df = (
    silver_df
    .withColumn(
        "late_days",
        datediff(col("ingestion_date"), col("txn_date"))
    )
    .withColumn(
        "is_late",
        when(col("late_days") > 0, True).otherwise(False)
    )
)

# Write Silver data as Delta
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

print("Silver layer created successfully.")

Bronze data loaded successfully.
Silver layer created successfully.


# STEP 5 — VALIDATE SILVER LAYER

In this step, we validate the cleaned Silver Delta table.

The validation checks include:

- Whether the Silver Delta path exists
- Total number of Silver records
- Silver table schema
- Sample cleaned transaction records
- Number of late-arriving transactions
- Number of duplicate transaction IDs

In [0]:
# STEP 5 — VALIDATE SILVER LAYER

# 1. Check Silver path
print("Silver path contents:")
display(dbutils.fs.ls(SILVER_PATH))


# 2. Read Silver Delta table
silver_check_df = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
)

print("Silver Delta table loaded successfully.")


# 3. Count Silver records
silver_record_count = silver_check_df.count()

print(f"Silver record count: {silver_record_count}")


# 4. Display Silver schema
print("\nSilver Schema:")
silver_check_df.printSchema()


# 5. Display sample cleaned records
print("Sample Silver Records:")
display(silver_check_df.limit(10))


# 6. Count late-arriving transactions
late_transaction_count = (
    silver_check_df
    .filter(col("is_late") == True)
    .count()
)

print(f"Late-arriving transaction count: {late_transaction_count}")


# 7. Check duplicate transaction IDs
duplicate_count = (
    silver_check_df
    .groupBy("txn_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"Duplicate transaction IDs: {duplicate_count}")

Silver path contents:


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/late_transaction/silver/_delta_log/,_delta_log/,0,1786364614592
dbfs:/Volumes/workspace/default/late_transaction/silver/part-00000-045c765d-a674-480d-92ee-bbf132c2c233.c000.snappy.parquet,part-00000-045c765d-a674-480d-92ee-bbf132c2c233.c000.snappy.parquet,27920,1786364555000


Silver Delta table loaded successfully.
Silver record count: 2000

Silver Schema:
root
 |-- txn_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- txn_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- late_days: integer (nullable = true)
 |-- is_late: boolean (nullable = true)

Sample Silver Records:


txn_id,user_id,txn_date,amount,ingestion_date,_rescued_data,late_days,is_late
1,427,2024-01-08,2353.0,2024-01-08,null,0,false
10,274,2024-01-18,1863.0,2024-01-18,null,0,false
100,441,2024-02-22,2552.0,2024-03-03,null,10,true
1000,203,2024-01-09,992.0,2024-01-24,null,15,true
1001,179,2024-01-07,3782.0,2024-01-07,null,0,false
1002,338,2024-01-21,1089.0,2024-01-26,null,5,true
1003,374,2024-01-23,3802.0,2024-01-24,null,1,true
1004,257,2024-01-30,1082.0,2024-02-01,null,2,true
1005,146,2024-01-11,430.0,2024-01-13,null,2,true
1006,210,2024-02-23,1315.0,2024-02-26,null,3,true


Late-arriving transaction count: 1415
Duplicate transaction IDs: 0


# STEP 6 — GOLD LAYER: DAILY REVENUE REPORT

The Gold layer creates a business-ready daily revenue report from the cleaned Silver transaction data.

The report includes:

- Transaction date
- Total number of transactions
- Total revenue
- Number of late-arriving transactions
- Revenue generated from late-arriving transactions

This Gold dataset is optimized for reporting and dashboard visualization.

In [0]:
# STEP 6 — CREATE GOLD DAILY REVENUE REPORT

from pyspark.sql.functions import (
    col,
    sum,
    count,
    when
)

# Define the Gold layer path.
GOLD_PATH = f"{BASE_PATH}/gold"

# Create the Gold directory if it does not exist.
dbutils.fs.mkdirs(GOLD_PATH)

# Read the cleaned Silver Delta data.
silver_df = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
)

print("Silver data loaded successfully.")


# Create the daily revenue report.
gold_revenue_df = (
    silver_df
    .groupBy("txn_date")
    .agg(
        count("txn_id").alias("total_transactions"),

        sum("amount").alias("total_revenue"),

        sum(
            when(col("is_late") == True, 1).otherwise(0)
        ).alias("late_transactions"),

        sum(
            when(col("is_late") == True, col("amount"))
            .otherwise(0)
        ).alias("late_transaction_revenue")
    )
    .orderBy("txn_date")
)


# Write the Gold report as a Delta table.
(
    gold_revenue_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_PATH)
)

print("Gold daily revenue report created successfully.")

Silver data loaded successfully.
Gold daily revenue report created successfully.


# STEP 7 — VALIDATE GOLD LAYER

The Gold layer contains the business-ready daily revenue report generated from the cleaned Silver transaction data.

In this step, we verify:

- Gold Delta path
- Total number of daily records
- Gold schema
- Daily revenue values
- Late transaction information

This validation ensures that the Gold layer is ready for reporting and dashboard consumption.

In [0]:
# STEP 7 — GOLD LAYER VALIDATION

# Read the Gold Delta table.
gold_check_df = (
    spark.read
    .format("delta")
    .load(GOLD_PATH)
)

print("Gold Delta table loaded successfully.")


# Count daily revenue records.
gold_record_count = gold_check_df.count()

print(f"Gold record count: {gold_record_count}")


# Display Gold schema.
print("\nGold Schema:")
gold_check_df.printSchema()


# Display the daily revenue report.
print("Daily Revenue Report:")
display(gold_check_df.orderBy("txn_date"))

Gold Delta table loaded successfully.
Gold record count: 60

Gold Schema:
root
 |-- txn_date: date (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- late_transactions: long (nullable = true)
 |-- late_transaction_revenue: double (nullable = true)

Daily Revenue Report:


txn_date,total_transactions,total_revenue,late_transactions,late_transaction_revenue
2024-01-01,28,56799.0,20,45276.0
2024-01-02,26,66741.0,23,56464.0
2024-01-03,35,93732.0,21,58120.0
2024-01-04,29,70961.0,20,54758.0
2024-01-05,26,69963.0,18,48066.0
2024-01-06,39,94410.0,28,61466.0
2024-01-07,35,95732.0,20,55980.0
2024-01-08,39,108769.0,28,82189.0
2024-01-09,35,88910.0,31,79185.0
2024-01-10,30,69564.0,21,44194.0


# STEP 8 — LATE TRANSACTION DETECTION

A late-arriving transaction is a transaction that is received after its actual transaction date.

In this project, a transaction is considered late when:

    ingestion_date > txn_date

The Silver layer already contains the `late_days` and `is_late` columns.

In this step, we identify all late-arriving transactions and calculate their count and revenue impact.

These transactions are important because they may change historical daily revenue reports.

In [0]:
# STEP 8 — IDENTIFY LATE-ARRIVING TRANSACTIONS

# Read the cleaned Silver Delta data.
silver_late_df = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
)

# Select only transactions identified as late.
late_transactions_df = (
    silver_late_df
    .filter(col("is_late") == True)
    .orderBy("txn_date")
)

# Count late transactions.
late_count = late_transactions_df.count()

print(f"Total late-arriving transactions: {late_count}")

# Display late transactions.
display(late_transactions_df)

Total late-arriving transactions: 1415


txn_id,user_id,txn_date,amount,ingestion_date,_rescued_data,late_days,is_late
945,134,2024-01-01,1867.0,2024-01-03,null,2,true
1554,253,2024-01-01,2499.0,2024-01-06,null,5,true
77,294,2024-01-01,2272.0,2024-01-06,null,5,true
911,333,2024-01-01,1757.0,2024-01-06,null,5,true
1477,374,2024-01-01,2133.0,2024-01-03,null,2,true
1679,247,2024-01-01,1011.0,2024-01-04,null,3,true
1455,324,2024-01-01,4344.0,2024-01-02,null,1,true
1869,398,2024-01-01,4281.0,2024-01-16,null,15,true
107,122,2024-01-01,1171.0,2024-01-04,null,3,true
1551,105,2024-01-01,1216.0,2024-01-11,null,10,true


# STEP 9 — AFFECTED HISTORICAL DATES

Late-arriving transactions can change revenue that was already reported for previous transaction dates.

Therefore, instead of recalculating the entire historical revenue dataset, we identify only the transaction dates affected by late-arriving transactions.

These affected dates will be used in the revenue correction process.

This targeted approach reduces unnecessary processing and improves pipeline efficiency.

In [0]:
# STEP 9 — FIND AFFECTED HISTORICAL DATES

# Get the unique transaction dates associated with late transactions.
affected_dates_df = (
    late_transactions_df
    .select("txn_date")
    .distinct()
    .orderBy("txn_date")
)

# Count the number of affected historical dates.
affected_date_count = affected_dates_df.count()

print(f"Number of affected historical dates: {affected_date_count}")

# Display the affected dates.
display(affected_dates_df)

Number of affected historical dates: 60


txn_date
2024-01-01
2024-01-02
2024-01-03
2024-01-04
2024-01-05
2024-01-06
2024-01-07
2024-01-08
2024-01-09
2024-01-10


# STEP 10 — RECALCULATE AFFECTED REVENUE

Late-arriving transactions can change revenue that was previously calculated for historical dates.

Instead of recalculating the complete Gold dataset, we recalculate revenue only for the affected transaction dates identified in Step 9.

The recalculated results will later be merged into the Gold Delta table using Delta Lake MERGE.

This targeted correction approach reduces unnecessary processing and supports efficient historical revenue correction.

In [0]:
# STEP 10 — RECALCULATE REVENUE FOR AFFECTED DATES

from pyspark.sql.functions import col, count, sum, when

# Recalculate revenue only for affected historical dates.
corrected_revenue_df = (
    silver_df
    .filter(col("txn_date").isin(
        [row["txn_date"] for row in affected_dates_df.collect()]
    ))
    .groupBy("txn_date")
    .agg(
        count("txn_id").alias("total_transactions"),

        sum("amount").alias("total_revenue"),

        sum(
            when(col("is_late") == True, 1).otherwise(0)
        ).alias("late_transactions"),

        sum(
            when(col("is_late") == True, col("amount"))
            .otherwise(0)
        ).alias("late_transaction_revenue")
    )
    .orderBy("txn_date")
)

print("Corrected revenue calculated for affected dates.")

display(corrected_revenue_df)

Corrected revenue calculated for affected dates.


txn_date,total_transactions,total_revenue,late_transactions,late_transaction_revenue
2024-01-01,28,56799.0,20,45276.0
2024-01-02,26,66741.0,23,56464.0
2024-01-03,35,93732.0,21,58120.0
2024-01-04,29,70961.0,20,54758.0
2024-01-05,26,69963.0,18,48066.0
2024-01-06,39,94410.0,28,61466.0
2024-01-07,35,95732.0,20,55980.0
2024-01-08,39,108769.0,28,82189.0
2024-01-09,35,88910.0,31,79185.0
2024-01-10,30,69564.0,21,44194.0


# STEP 11 — DELTA LAKE MERGE

The corrected revenue calculated for affected historical dates must be applied to the existing Gold revenue table.

Delta Lake MERGE is used to perform this correction.

If an affected transaction date already exists in the Gold table, its revenue values are updated with the corrected values.

If a transaction date does not exist, a new record is inserted.

This allows historical revenue corrections to be performed efficiently without rebuilding the complete Gold dataset.

In [0]:
# STEP 11 — APPLY CORRECTIONS USING DELTA MERGE

from delta.tables import DeltaTable

# Load the existing Gold Delta table.
gold_delta_table = DeltaTable.forPath(
    spark,
    GOLD_PATH
)

# Merge corrected revenue into the Gold table.
(
    gold_delta_table.alias("gold")
    .merge(
        corrected_revenue_df.alias("corrected"),
        "gold.txn_date = corrected.txn_date"
    )
    .whenMatchedUpdate(set={
        "total_transactions": "corrected.total_transactions",
        "total_revenue": "corrected.total_revenue",
        "late_transactions": "corrected.late_transactions",
        "late_transaction_revenue": "corrected.late_transaction_revenue"
    })
    .whenNotMatchedInsert(values={
        "txn_date": "corrected.txn_date",
        "total_transactions": "corrected.total_transactions",
        "total_revenue": "corrected.total_revenue",
        "late_transactions": "corrected.late_transactions",
        "late_transaction_revenue": "corrected.late_transaction_revenue"
    })
    .execute()
)

print("Gold revenue corrected successfully using Delta MERGE.")

Gold revenue corrected successfully using Delta MERGE.


# STEP 12 — VERIFY CORRECTED GOLD REVENUE

The Gold Delta table has been updated using Delta Lake MERGE.

In this step, we reload the Gold table and verify that the corrected revenue values are available for the affected historical dates.

This confirms that the historical revenue correction was successfully applied.

In [0]:
# STEP 12 — VERIFY GOLD AFTER DELTA MERGE

# Reload the Gold Delta table after the MERGE operation.
gold_verified_df = (
    spark.read
    .format("delta")
    .load(GOLD_PATH)
)

print("Gold Delta table loaded successfully after MERGE.")

# Display corrected values only for affected dates.
corrected_gold_df = (
    gold_verified_df
    .join(
        affected_dates_df,
        on="txn_date",
        how="inner"
    )
    .orderBy("txn_date")
)

print("Corrected revenue for affected dates:")
display(corrected_gold_df)

Gold Delta table loaded successfully after MERGE.
Corrected revenue for affected dates:


txn_date,total_transactions,total_revenue,late_transactions,late_transaction_revenue
2024-01-01,28,56799.0,20,45276.0
2024-01-02,26,66741.0,23,56464.0
2024-01-03,35,93732.0,21,58120.0
2024-01-04,29,70961.0,20,54758.0
2024-01-05,26,69963.0,18,48066.0
2024-01-06,39,94410.0,28,61466.0
2024-01-07,35,95732.0,20,55980.0
2024-01-08,39,108769.0,28,82189.0
2024-01-09,35,88910.0,31,79185.0
2024-01-10,30,69564.0,21,44194.0


# STEP 13 — DATA QUALITY CHECKS

Data quality checks ensure that the Silver transaction data is reliable before it is used for revenue reporting.

The following checks are performed:

- Null transaction IDs
- Duplicate transaction IDs
- Invalid or negative transaction amounts
- Null transaction dates
- Null ingestion dates
- Invalid late-arriving transaction calculations

These checks help ensure that the Gold revenue report is based on valid and consistent transaction data.

In [0]:
# STEP 13 — DATA QUALITY CHECKS

from pyspark.sql.functions import col

# Read the final Silver Delta data.
quality_df = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
)

# --------------------------------------------------
# 1. NULL TRANSACTION IDs
# --------------------------------------------------

null_txn_ids = (
    quality_df
    .filter(col("txn_id").isNull())
    .count()
)

# --------------------------------------------------
# 2. DUPLICATE TRANSACTION IDs
# --------------------------------------------------

duplicate_txn_ids = (
    quality_df
    .groupBy("txn_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

# --------------------------------------------------
# 3. NEGATIVE AMOUNTS
# --------------------------------------------------

negative_amounts = (
    quality_df
    .filter(col("amount") < 0)
    .count()
)

# --------------------------------------------------
# 4. NULL TRANSACTION DATES
# --------------------------------------------------

null_txn_dates = (
    quality_df
    .filter(col("txn_date").isNull())
    .count()
)

# --------------------------------------------------
# 5. NULL INGESTION DATES
# --------------------------------------------------

null_ingestion_dates = (
    quality_df
    .filter(col("ingestion_date").isNull())
    .count()
)

# --------------------------------------------------
# 6. INVALID LATE TRANSACTION FLAGS
# --------------------------------------------------

invalid_late_flags = (
    quality_df
    .filter(
        (
            (col("ingestion_date") > col("txn_date")) &
            (col("is_late") != True)
        )
        |
        (
            (col("ingestion_date") <= col("txn_date")) &
            (col("is_late") != False)
        )
    )
    .count()
)

# --------------------------------------------------
# DISPLAY DATA QUALITY RESULTS
# --------------------------------------------------

print("========== DATA QUALITY REPORT ==========")

print(f"Null transaction IDs       : {null_txn_ids}")
print(f"Duplicate transaction IDs  : {duplicate_txn_ids}")
print(f"Negative amounts           : {negative_amounts}")
print(f"Null transaction dates     : {null_txn_dates}")
print(f"Null ingestion dates       : {null_ingestion_dates}")
print(f"Invalid late flags         : {invalid_late_flags}")

print("==========================================")

# Overall quality status
if (
    null_txn_ids == 0
    and duplicate_txn_ids == 0
    and negative_amounts == 0
    and null_txn_dates == 0
    and null_ingestion_dates == 0
    and invalid_late_flags == 0
):
    print("DATA QUALITY STATUS: PASSED")
else:
    print("DATA QUALITY STATUS: FAILED")

========== DATA QUALITY REPORT ==========
Null transaction IDs       : 0
Duplicate transaction IDs  : 0
Negative amounts           : 0
Null transaction dates     : 0
Null ingestion dates       : 0
Invalid late flags         : 0
DATA QUALITY STATUS: PASSED


# STEP 14 — WATERMARK & INCREMENTAL PROCESSING

A watermark helps control how long Spark keeps state for previously processed streaming data.

In this project, the watermark is applied to the transaction date while processing the incoming transaction stream.

It helps limit the amount of historical state maintained by the streaming query and supports scalable incremental processing.

Late-arriving transactions within the allowed watermark period can still be processed.

In [0]:
# STEP 14 — WATERMARK CONFIGURATION

from pyspark.sql.functions import col

# Create an incremental streaming DataFrame from the source.
watermarked_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option(
            "cloudFiles.schemaLocation",
            f"{BASE_PATH}/checkpoints/watermark_schema"
        )
        .option("header", "true")
        .option("inferSchema", "true")
        .load(SOURCE_PATH)

        # Watermark allows Spark to manage late-arriving data
        # while limiting the amount of streaming state retained.
        .withWatermark("txn_date", "7 days")
)

print("Watermark configured successfully.")
print("Watermark threshold: 7 days")

Watermark configured successfully.
Watermark threshold: 7 days
